In [47]:
from src.analysis.processor import PreselProcessor
import uproot
from config.customEvtSel import LoosetwoTau
import operator as opr
import awkward as ak

In [3]:
issue_file = "root://cmseos.fnal.gov//store/user/joyzhou/vbfskim_hadded/2022PostEE/TTbar/TTtoLNu2Q_5.root"

In [4]:
events = uproot.open(f'{issue_file}:Events').arrays()

In [11]:
evtsel = LoosetwoTau(is_mc=True)
evtsel._setobjsel(events)

In [12]:
tau_masker = evtsel.getObjMasker(events, "Tau")

base_conditions = {
    'pt': (opr.ge,),
    'eta': (opr.le, abs),
    'dz': (opr.lt, abs),
    'idvsjet': (opr.ge,),
    'idvsmu': (opr.ge,),
    'idvse': (opr.ge,)
    }
tau_mask = tau_masker.create_combined_mask(base_conditions)
tau_nummask = tau_masker.numselmask(tau_mask, opr.ge)
tau_masker, events = evtsel.selobjhelper(events, '>= 2 Medium hadronic Taus', tau_masker, tau_nummask)

In [19]:
tau_mask = tau_masker.create_combined_mask(base_conditions)
tau_proc = evtsel.getObjProc('Tau')

In [22]:
dr_mask, sort_mask = tau_proc.dRwSelf(events, 0.5, tau_mask)

In [23]:
dr_mask_events = tau_masker.maskredmask(dr_mask, opr.ge, 1)

In [26]:
len(events) == len(dr_mask_events)

True

In [52]:
len(sort_mask)

2045

In [27]:
tau_mask = tau_mask[sort_mask][dr_mask_events]

In [57]:
ak.all(ak.num(tau_mask, axis=1) >= 2)

True

In [33]:
filtered = events[dr_mask_events]

In [34]:
len(filtered) == len(tau_mask)

True

In [65]:
zipped = tau_proc.set_zipped(filtered, tau_proc._name, tau_proc._mapcfg)

In [75]:
zipped.pt[0:5]

<Array [[57.1, 60.4], [190, ...], ..., [65.9, 43.8]] type='5 * var * float32'>

In [73]:
tau_proc.sortmask(zipped.pt)

<Array [[1, 0], [0, 1], ..., [1, ..., 2], [0, 1, 2]] type='2023 * var * int64'>

In [79]:
tau_mask[0:14]

<Array [[True, True], [True, ...], ..., [True, True]] type='14 * var * bool'>

In [77]:
ak.num(zipped[0:5], axis=-1)

<Array [2, 2, 3, 3, 2] type='5 * int64'>

In [66]:
ak.all(ak.num(zipped, axis=1) >= 2)

True

In [69]:
ak.all(ak.num(zipped[tau_mask], axis=1) >=2)

False

In [38]:
new_dr = dr_mask[dr_mask_events]

In [62]:
ak.all(ak.num(zipped, axis=1) >= 2)

False